In [32]:
#ALPEREN POLAT 220709038
# Import necessary fundamental libraries that ai recommended
import os
import re
import random
import requests
import pickle
import nltk
from google.colab import drive

# Download required NLTK data package for word tokenization
nltk.download('punkt')
nltk.download('punkt_tab') # Required for newer NLTK versions in Colab
from nltk.tokenize import word_tokenize

# Mount Google Drive to the Colab environment
drive.mount('/content/drive')

# Define the base directory for the project
base = '/content/drive/MyDrive/nlp_project'

# Create the required folder structure: data, vocab, and models
# exist_ok=True prevents errors if the directory already exists
for folder in ['data', 'vocab', 'models']:
    os.makedirs(f'{base}/{folder}', exist_ok=True) # Note: Creates directories dynamically

print("Folder structure created successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder structure created successfully!


In [33]:
import re
import random
import requests
from nltk.tokenize import word_tokenize

# --- STEP 2: FUNCTIONS ---

def download_book(book_id):
    """
    Downloads the raw text of a book using its Gutenberg ID.
    Added a User-Agent header to prevent Gutenberg from blocking the request.
    """
    url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    # Using a User-Agent mimics a real browser, preventing 403 Forbidden errors
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return response.text
    else:
        print(f"Warning: Failed to download book with ID {book_id}. Status Code: {response.status_code}")
        return ""

def clean_gutenberg_text(text):
    """
    Removes the header and footer boilerplate text added by Gutenberg.
    """
    pattern = r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*(.*?)\*\*\* END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*"
    match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)

    if match:
        return match.group(1)
    else:
        return text

def chunk_text(text, chunk_size=500):
    """
    Splits the cleaned text into passages of exactly the specified number of words (500).
    """
    tokens = word_tokenize(text)
    chunks = []

    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i : i + chunk_size]

        # Append to the list only if the chunk has exactly 500 words
        if len(chunk_tokens) == chunk_size:
            chunks.append(" ".join(chunk_tokens))

    # CRITICAL: This return statement must be outside the for loop!
    return chunks


# --- STEP 3: MAIN EXECUTION ---

# Author names and their corresponding Gutenberg book IDs
author_books = {
    "Mark Twain": [74, 76, 1837, 86, 245],
    "Jane Austen": [1342, 161, 158, 121, 105],
    "Arthur Conan Doyle": [2852, 244, 2097, 1661, 834],
    "H.G. Wells": [35, 36, 5230, 159, 1013]
}

passages = {}
author_to_id = {"Mark Twain": 0, "Jane Austen": 1, "Arthur Conan Doyle": 2, "H.G. Wells": 3}

print("Downloading and processing data. This may take 1-2 minutes...")

for author, book_ids in author_books.items():
    author_chunks = []

    for book_id in book_ids:
        raw_text = download_book(book_id)
        clean_text = clean_gutenberg_text(raw_text)
        chunks = chunk_text(clean_text, chunk_size=500)

        # Extend the list with the newly created chunks
        author_chunks.extend(chunks)

    # Cap each author's class at 200 chunks to ensure classes are balanced
    random.seed(42) # Set seed for reproducibility
    random.shuffle(author_chunks)

    # Safety check: warn if an author didn't reach 200 chunks
    if len(author_chunks) < 200:
        print(f"WARNING: {author} only has {len(author_chunks)} chunks! (Expected 200)")

    passages[author] = author_chunks[:200]
    print(f"Created a total of {len(passages[author])} 500-word passages for {author}.")

# Splitting the data into 70% Train, 10% Validation, and 20% Test
train_corpus, train_labels = [], []
val_corpus, val_labels = [], []
test_corpus, test_labels = [], []

for author, chunks in passages.items():
    label = author_to_id[author]

    train_chunks = chunks[:140]
    val_chunks = chunks[140:160]
    test_chunks = chunks[160:200]

    train_corpus.extend(train_chunks)
    train_labels.extend([label] * len(train_chunks))

    val_corpus.extend(val_chunks)
    val_labels.extend([label] * len(val_chunks))

    test_corpus.extend(test_chunks)
    test_labels.extend([label] * len(test_chunks))

def shuffle_data(corpus, labels):
    combined = list(zip(corpus, labels))
    random.shuffle(combined)
    return zip(*combined)

train_corpus, train_labels = shuffle_data(train_corpus, train_labels)
val_corpus, val_labels = shuffle_data(val_corpus, val_labels)
test_corpus, test_labels = shuffle_data(test_corpus, test_labels)

train_data = (list(train_corpus), list(train_labels))
val_data = (list(val_corpus), list(val_labels))
test_data = (list(test_corpus), list(test_labels))

print("\nData stratified splitting process completed!")
print(f"Train size: {len(train_data[0])}")
print(f"Validation size: {len(val_data[0])}")
print(f"Test size: {len(test_data[0])}")

Created a total of 200 500-word passages for Mark Twain.
Created a total of 200 500-word passages for Jane Austen.
Created a total of 200 500-word passages for Arthur Conan Doyle.
Created a total of 200 500-word passages for H.G. Wells.

Data stratified splitting process completed!
Train size: 560
Validation size: 80
Test size: 160


In [34]:
# Saving the processed data files
# The "wb" parameter stands for Write Binary

with open(f'{base}/data/passages.pkl', 'wb') as f:
    pickle.dump(passages, f) # raw dictionary: {author: [texts]} format

with open(f'{base}/data/train.pkl', 'wb') as f:
    pickle.dump(train_data, f) # tuple format: (corpus, labels)

with open(f'{base}/data/val.pkl', 'wb') as f:
    pickle.dump(val_data, f)

with open(f'{base}/data/test.pkl', 'wb') as f:
    pickle.dump(test_data, f)

print("All data successfully saved to Google Drive!")

All data successfully saved to Google Drive!


In [35]:
import math
import numpy as np
from collections import Counter
import pickle

def build_vocabulary(corpus):
    """
    Scans the training corpus and assigns a unique integer index to each unique word.
    Returns a dictionary mapping words to their indices.
    """
    vocab = {}
    idx = 0
    for doc in corpus:
        # Our chunks were joined by spaces in the previous step
        for word in doc.split():
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

def compute_idf(corpus, vocab):
    """
    Computes the Inverse Document Frequency (IDF) for each word in the vocabulary.
    Formula: IDF = log(Total Documents / Documents containing the word)
    """
    N = len(corpus)
    df = Counter() # Document Frequency

    for doc in corpus:
        # Use a set to count unique words per document (we only care IF it appears, not how many times)
        words_in_doc = set(doc.split())
        for word in words_in_doc:
            if word in vocab:
                df[word] += 1

    idf = {}
    for word, _ in vocab.items():
        # Calculate log probability. df[word] will never be 0 because the word is in the vocab derived from the corpus.
        idf[word] = math.log(N / df[word])

    return idf

def tfidf_vectorize(doc, vocab, idf):
    """
    Converts a string of text into a TF-IDF numerical vector using numpy.
    """
    # Initialize a vector of zeros with the size of our vocabulary
    vec = np.zeros(len(vocab))
    words = doc.split()

    # Calculate Term Frequency (TF)
    tf = Counter(words)

    for word, count in tf.items():
        if word in vocab:
            idx = vocab[word]
            # TF-IDF Score = Raw Count * IDF Score
            vec[idx] = count * idf[word]

    return vec

# --- Mathematical Operations ---

def dot(v1, v2):
    return np.dot(v1, v2)

def norm(v):
    return np.linalg.norm(v)

def cosine(v1, v2):
    """
    Calculates the cosine similarity between two vectors.
    """
    n1, n2 = norm(v1), norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return dot(v1, v2) / (n1 * n2)

# --- Centroid and Prediction Logic ---

def compute_centroids(corpus, labels, vocab, idf, num_classes=4):
    """
    Calculates the mean TF-IDF vector (centroid) for each author class.
    """
    # 1. Convert all training documents to TF-IDF vectors
    vectors = []
    for doc in corpus:
        vectors.append(tfidf_vectorize(doc, vocab, idf))

    vectors = np.array(vectors)
    labels = np.array(labels)

    centroids = {}
    for class_id in range(num_classes):
        # 2. Extract only the vectors belonging to the current author
        class_vectors = vectors[labels == class_id]
        # 3. Calculate the mean across the column axis (axis=0)
        centroids[class_id] = np.mean(class_vectors, axis=0)

    return centroids

def vsm_predict(doc, vocab, idf, centroids):
    """
    Predicts the author by finding the centroid with the highest cosine similarity to the document.
    """
    vec = tfidf_vectorize(doc, vocab, idf)

    best_class = -1
    max_sim = -float('inf') # Start with the lowest possible similarity

    # Compare the new document vector to each author's centroid
    for class_id, centroid_vec in centroids.items():
        sim = cosine(vec, centroid_vec)
        if sim > max_sim:
            max_sim = sim
            best_class = class_id

    return best_class

In [36]:
import time

base = '/content/drive/MyDrive/nlp_project'

# Load the previously saved training and validation data
with open(f'{base}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

with open(f'{base}/data/val.pkl', 'rb') as f:
    val_corpus, val_labels = pickle.load(f)

print("Starting VSM Training Pipeline...")
start_time = time.time()

# 1. Build Vocabulary
print("Building vocabulary...")
word2idx = build_vocabulary(train_corpus)
print(f"Vocabulary size: {len(word2idx)} unique words.")

# 2. Compute IDF Scores
print("Computing IDF scores...")
idf_scores = compute_idf(train_corpus, word2idx)

# 3. Compute Centroids
print("Computing class centroids...")
centroids = compute_centroids(train_corpus, train_labels, word2idx, idf_scores, num_classes=4)

vsm_training_time = time.time() - start_time
print(f"VSM Training completed in {vsm_training_time:.2f} seconds.")

# 4. Evaluate on Validation Set
print("\nEvaluating on Validation Set...")
correct_predictions = 0

for doc, true_label in zip(val_corpus, val_labels):
    pred_label = vsm_predict(doc, word2idx, idf_scores, centroids)
    if pred_label == true_label:
        correct_predictions += 1

val_accuracy = (correct_predictions / len(val_labels)) * 100
print(f"--> VSM Validation Accuracy: {val_accuracy:.2f}%")

# 5. Save the models and vocabulary exactly as specified in the assignment requirements
# Save the vocabulary mapping [cite: 70]
with open(f'{base}/vocab/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

# Save the VSM data (centroids and idf scores) as a dictionary [cite: 73, 94]
vsm_data = {'centroids': centroids, 'idf': idf_scores}
with open(f'{base}/models/vsm.pkl', 'wb') as f:
    pickle.dump(vsm_data, f)

print("\nVSM model (vsm.pkl) and vocabulary (word2idx.pkl) successfully saved to Google Drive!")

Starting VSM Training Pipeline...
Building vocabulary...
Vocabulary size: 19541 unique words.
Computing IDF scores...
Computing class centroids...
VSM Training completed in 0.46 seconds.

Evaluating on Validation Set...
--> VSM Validation Accuracy: 96.25%

VSM model (vsm.pkl) and vocabulary (word2idx.pkl) successfully saved to Google Drive!


In [37]:
import math
from collections import Counter
import pickle

def build_lm_vocab(corpus):
    """
    # Tüm corpus'u (metin havuzunu) tarayarak N-gram modeli için kelime dağarcığını (vocabulary) oluşturur.
    """
    vocab = set()
    for doc in corpus:
        for word in doc.split():
            vocab.add(word)
    return vocab

def prepare_tokens(doc):
    """
    # Metni kelimelere ayırır ve N-gram pencereleri için başına/sonuna başlangıç ve bitiş tokenları ekler.
    """
    # Cümle başı ve sonunu belirtmek için <s> ve </s> tokenlarını kullanıyoruz
    tokens = ['<s>'] + doc.split() + ['</s>']
    return tokens

def extract_ngrams(tokens, n=2):
    """
    # Verilen token listesinden n-gram (ikili, üçlü vb.) grupları çıkarır.
    """
    ngrams = []
    # Token listesi üzerinde n boyutunda bir pencere kaydırıyoruz
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i : i + n]))
    return ngrams

def build_ngram_counts(corpus):
    """
    # Eğitim verisindeki unigram (tekil) ve bigram (ikili) kelime frekanslarını hesaplar.
    """
    unigram_counts = Counter()
    bigram_counts = Counter()

    for doc in corpus:
        tokens = prepare_tokens(doc)

        # Unigram (1-gram) frekansları
        unigrams = extract_ngrams(tokens, 1)
        unigram_counts.update(unigrams)

        # Bigram (2-gram) frekansları
        bigrams = extract_ngrams(tokens, 2)
        bigram_counts.update(bigrams)

    return unigram_counts, bigram_counts

def ngram_prob(ngram, ngram_counts, n_minus_1_gram_counts, vocab_size, k=0.1):
    """
    # Add-k smoothing (k-eklemeli yumuşatma) kullanarak n-gram olasılığını hesaplar.
    # Bu yöntem, eğitim verisinde hiç geçmeyen kelime grupları için 0 olasılık hatasını (Zero Probability Problem) önler.
    """
    # N-gram'ın geçme sayısı
    count_ngram = ngram_counts.get(ngram, 0)

    # N-1 gram'ın geçme sayısı (Örneğin bigram için, ilk kelimenin geçme sayısı)
    n_minus_1_gram = ngram[:-1]
    count_n_minus_1 = n_minus_1_gram_counts.get(n_minus_1_gram, 0)

    # Olasılık formülü: (C(w1, w2) + k) / (C(w1) + k * V)
    prob = (count_ngram + k) / (count_n_minus_1 + k * vocab_size)
    return prob

def interpolated_log_prob(bigram, bigram_counts, unigram_counts, total_unigrams, vocab_size, lam, k=0.1):
    """
    # Interpolasyon kullanarak logaritmik olasılığı hesaplar: \lambda * Bigram_Prob + (1-\lambda) * Unigram_Prob
    # lam (lambda) parametresi, iki model (Bigram ve Unigram) arasında ne kadar ağırlık verileceğini belirler.
    """
    w1 = (bigram[0],)

    # Bigram olasılığı (Add-k yumuşatılmış)
    p_bigram = ngram_prob(bigram, bigram_counts, unigram_counts, vocab_size, k)

    # Unigram olasılığı (Add-k yumuşatılmış)
    # total_unigrams: corpus'taki toplam kelime sayısı
    p_unigram = (unigram_counts.get((bigram[1],), 0) + k) / (total_unigrams + k * vocab_size)

    # İki olasılığı lambda ile ağırlıklandırarak birleştir (Interpolation)
    p_interp = (lam * p_bigram) + ((1 - lam) * p_unigram)

    # Çarpım işlemlerinde sayıların aşırı küçülmesini (underflow) önlemek için logaritma alıyoruz
    return math.log(p_interp)

def document_log_prob(doc, bigram_counts, unigram_counts, total_unigrams, vocab_size, lam, k=0.1):
    """
    # Bir belgenin tamamının (tüm bigramlarının toplamı) logaritmik olasılığını hesaplar.
    """
    tokens = prepare_tokens(doc)
    bigrams = extract_ngrams(tokens, 2)

    log_prob = 0.0
    for bg in bigrams:
        # Logaritmada olasılıklar çarpılmaz, toplanır
        log_prob += interpolated_log_prob(bg, bigram_counts, unigram_counts, total_unigrams, vocab_size, lam, k)

    return log_prob

def lm_predict(doc, class_data, vocab_size, lam, k=0.1):
    """
    # Belgenin her yazar sınıfı için olasılığını hesaplar ve Bayes Teoremini kullanarak en yüksek olasılığa sahip sınıfı döndürür.
    """
    best_class = -1
    max_log_prob = -float('inf')

    for class_id, data in class_data.items():
        # Sözlükten o yazar sınıfına ait model verilerini çekiyoruz
        bg_counts = data['bigram_counts']
        ug_counts = data['unigram_counts']
        total_ug = data['total_unigrams']
        prior = data['log_prior']

        # P(Document | Class) hesaplaması
        doc_lp = document_log_prob(doc, bg_counts, ug_counts, total_ug, vocab_size, lam, k)

        # Bayes Teoremi (Logaritma alanında): P(Class | Document) = log P(Class) + log P(Document | Class)
        total_lp = prior + doc_lp

        if total_lp > max_log_prob:
            max_log_prob = total_lp
            best_class = class_id

    return best_class

<>:71: SyntaxWarning: invalid escape sequence '\l'
<>:71: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_21563/2481356794.py:71: SyntaxWarning: invalid escape sequence '\l'
  # Interpolasyon kullanarak logaritmik olasılığı hesaplar: \lambda * Bigram_Prob + (1-\lambda) * Unigram_Prob


In [38]:
import time

base = '/content/drive/MyDrive/nlp_project'

# Önceden kaydettiğimiz eğitim ve doğrulama setlerini yüklüyoruz
with open(f'{base}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)

with open(f'{base}/data/val.pkl', 'rb') as f:
    val_corpus, val_labels = pickle.load(f)

print("Starting N-gram LM Training Pipeline...")
start_time = time.time()

# 1. Ortak Kelime Dağarcığını (Vocabulary) Oluştur
print("Building LM vocabulary...")
lm_vocab = build_lm_vocab(train_corpus)
vocab_size = len(lm_vocab)
print(f"LM Vocabulary size: {vocab_size} unique words.")

# Sınıfların frekanslarını ve verilerini tutacağımız sözlükler
class_ngram_data = {}
class_unigram_data = {}
class_log_prior = {}
total_docs = len(train_labels)

# Yazar ID'leri üzerinden döngü başlatıyoruz (0: Twain, 1: Austen, 2: Doyle, 3: Wells)
class_data = {}

for class_id in range(4):
    # Sadece o yazarın (sınıfın) belgelerini filtreliyoruz
    class_docs = [doc for doc, label in zip(train_corpus, train_labels) if label == class_id]

    # Yazarın kelime frekanslarını (unigram ve bigram) saydırıyoruz
    ug_counts, bg_counts = build_ngram_counts(class_docs)

    # Toplam unigram sayısı
    total_ug = sum(ug_counts.values())

    # Prior (Önsel) Olasılık: log(Yazarın Belge Sayısı / Toplam Belge Sayısı)
    # Her sınıftan 140 belgemiz olduğu için aslında hepsi log(0.25) değerini alacak
    prior = math.log(len(class_docs) / total_docs)

    # Tüm hesaplamaları hocanın talimatlarına uygun isimlendirmelerle saklıyoruz
    class_ngram_data[class_id] = bg_counts
    class_unigram_data[class_id] = ug_counts
    class_log_prior[class_id] = prior

    # Tahmin fonksiyonuna (predict) kolay parametre göndermek için paketliyoruz
    class_data[class_id] = {
        'bigram_counts': bg_counts,
        'unigram_counts': ug_counts,
        'total_unigrams': total_ug,
        'log_prior': prior
    }

lm_training_time = time.time() - start_time
print(f"LM Training completed in {lm_training_time:.2f} seconds.")

# 2. Doğrulama Setinde (Validation Set) Hyperparameter Tuning İşlemi
print("\nTuning Lambda parameter on Validation Set...")

best_lam = 0.0
best_acc = 0.0
# En iyi ağırlığı bulmak için farklı lambda değerlerini deniyoruz
lambdas_to_test = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

for lam in lambdas_to_test:
    correct = 0
    for doc, true_label in zip(val_corpus, val_labels):
        pred_label = lm_predict(doc, class_data, vocab_size, lam=lam, k=0.1)
        if pred_label == true_label:
            correct += 1

    acc = (correct / len(val_labels)) * 100
    print(f"Lambda = {lam} --> Validation Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        best_lam = lam

print(f"\nBest Lambda parameter found: {best_lam} with {best_acc:.2f}% accuracy.")

# 3. İstenen Dosyaları Google Drive'a Kaydetme
# LM Kelime Dağarcığını kaydet
with open(f'{base}/vocab/lm_vocab.pkl', 'wb') as f:
    pickle.dump(lm_vocab, f)

# N-gram Modeli Sözlüğünü hocanın istediği anahtarlarla (keys) formatlayıp kaydet
ngram_model_data = {
    'class_ngram_data': class_ngram_data,
    'class_unigram_data': class_unigram_data,
    'class_log_prior': class_log_prior,
    'best_lam': best_lam
}

with open(f'{base}/models/ngram.pkl', 'wb') as f:
    pickle.dump(ngram_model_data, f)

print("\nN-gram model (ngram.pkl) and vocabulary (lm_vocab.pkl) successfully saved to Google Drive!")

Starting N-gram LM Training Pipeline...
Building LM vocabulary...
LM Vocabulary size: 19541 unique words.
LM Training completed in 0.57 seconds.

Tuning Lambda parameter on Validation Set...
Lambda = 0.1 --> Validation Accuracy: 98.75%
Lambda = 0.3 --> Validation Accuracy: 98.75%
Lambda = 0.5 --> Validation Accuracy: 98.75%
Lambda = 0.7 --> Validation Accuracy: 97.50%
Lambda = 0.9 --> Validation Accuracy: 97.50%
Lambda = 1.0 --> Validation Accuracy: 97.50%

Best Lambda parameter found: 0.1 with 98.75% accuracy.

N-gram model (ngram.pkl) and vocabulary (lm_vocab.pkl) successfully saved to Google Drive!


In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pickle
from collections import Counter

base = '/content/drive/MyDrive/nlp_project'

# 1. Gerekli verileri ve VSM kelime dağarcığını/ağırlıklarını yüklüyoruz
print("Loading data and VSM artifacts...")
with open(f'{base}/data/train.pkl', 'rb') as f:
    train_corpus, train_labels = pickle.load(f)
with open(f'{base}/data/val.pkl', 'rb') as f:
    val_corpus, val_labels = pickle.load(f)

with open(f'{base}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)
with open(f'{base}/models/vsm.pkl', 'rb') as f:
    vsm_data = pickle.load(f)
    idf_scores = vsm_data['idf']

# 2. Metinleri TF-IDF vektörlerine ve PyTorch Tensörlerine dönüştürme fonksiyonu
def text_to_tensor(corpus, labels, vocab, idf):
    """
    # Bu fonksiyon VSM'deki mantıkla aynıdır, ancak çıkışı sinir ağının işleyebileceği PyTorch tensörüdür.
    """
    vectors = []
    for doc in corpus:
        vec = np.zeros(len(vocab))
        words = doc.split()
        tf = Counter(words)
        for word, count in tf.items():
            if word in vocab:
                idx = vocab[word]
                # TF-IDF Skorunu hesapla ve vektöre yerleştir
                vec[idx] = count * idf[word]
        vectors.append(vec)

    # float32: Ağırlık güncellemeleri için PyTorch'un beklediği standart ondalık tipi
    # long: Etiketler (0,1,2,3) tamsayı olduğu için long tipi kullanılır
    X = torch.tensor(np.array(vectors), dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y

# Tensörleri oluştur
print("Preparing tensors for MLP. This might take a few seconds...")
X_train, y_train = text_to_tensor(train_corpus, train_labels, word2idx, idf_scores)
X_val, y_val = text_to_tensor(val_corpus, val_labels, word2idx, idf_scores)
print(f"Train tensor shape: {X_train.shape}")

# 3. MLP Model Mimarisi
class AuthorClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(AuthorClassifierMLP, self).__init__()
        # İlk tam bağlı (fully connected) katman
        self.fc1 = nn.Linear(input_size, hidden_size)
        # Aktivasyon fonksiyonu (Doğrusal olmayan kalıpları öğrenmek için ReLU kullanıyoruz)
        self.relu = nn.ReLU()
        # Çıktı katmanı (4 yazar için sınıf skorlarını üretir)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Verinin katmanlardan geçiş sırası (İleri besleme)
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

Loading data and VSM artifacts...
Preparing tensors for MLP. This might take a few seconds...
Train tensor shape: torch.Size([560, 19541])


In [40]:
import time

# Hiperparametreler (Ağın öğrenme kuralları)
input_size = len(word2idx) # Girdi boyutu kelime dağarcığımız kadar olmalı
hidden_size = 128          # Gizli katmandaki nöron sayısı
num_classes = 4            # 4 farklı yazar sınıfımız var
learning_rate = 0.001      # Ağın öğrenme hızı (adımların büyüklüğü)
num_epochs = 20            # Eğitimin kaç tur döneceği

# Modeli başlat
model = AuthorClassifierMLP(input_size, hidden_size, num_classes)

# Kayıp fonksiyonu (Çok sınıflı sınıflandırma görevleri için Cross Entropy en iyisidir)
criterion = nn.CrossEntropyLoss()
# Optimizasyon algoritması
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Starting MLP Training...")
start_time = time.time()

# Eğitim Döngüsü (Training Loop)
for epoch in range(num_epochs):
    # İleri besleme (Forward pass)
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Geri yayılım ve optimizasyon (Backward pass)
    optimizer.zero_grad() # Önceki turun gradyanlarını (hatalarını) sıfırla
    loss.backward()       # Hataları geriye doğru hesapla
    optimizer.step()      # Ağırlıkları güncelle

    # Her 5 adımda bir kayıp (loss) değerini ekrana yazdır
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

mlp_training_time = time.time() - start_time
print(f"MLP Training completed in {mlp_training_time:.2f} seconds.")

# Doğrulama (Validation) Setinde Değerlendirme
model.eval() # Modeli değerlendirme (evaluation) moduna al
with torch.no_grad(): # Gradyan hesaplamalarını kapat (hafıza tasarrufu ve hız için)
    outputs = model(X_val)

    # Her metin için en yüksek skora (olasılığa) sahip olan yazar ID'sini seç
    _, predicted = torch.max(outputs.data, 1)

    # Doğru tahminlerin toplamını al
    correct = (predicted == y_val).sum().item()
    val_accuracy = (correct / len(y_val)) * 100
    print(f"--> MLP Validation Accuracy: {val_accuracy:.2f}%")

# Modeli Google Drive'a kaydet (Tüm ağırlıklarıyla birlikte PyTorch nesnesi olarak)
torch.save(model, f'{base}/models/mlp.pt')
print("\nPyTorch MLP model (mlp.pt) successfully saved to Google Drive!")

Starting MLP Training...
Epoch [5/20], Loss: 0.0123
Epoch [10/20], Loss: 0.0001
Epoch [15/20], Loss: 0.0000
Epoch [20/20], Loss: 0.0000
MLP Training completed in 2.65 seconds.
--> MLP Validation Accuracy: 100.00%

PyTorch MLP model (mlp.pt) successfully saved to Google Drive!


In [41]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import time
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd

base = '/content/drive/MyDrive/nlp_project'

# --- 1. MLP Sınıf Tanımı (Modeli yükleyebilmek için sınıfın bellekte olması gerekir) ---
class AuthorClassifierMLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(AuthorClassifierMLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# --- 2. Verileri ve Modelleri Yükleme ---
print("Loading test data and models...")
with open(f'{base}/data/test.pkl', 'rb') as f:
    test_corpus, test_labels = pickle.load(f)

with open(f'{base}/vocab/word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)
with open(f'{base}/models/vsm.pkl', 'rb') as f:
    vsm_data = pickle.load(f)
    centroids = vsm_data['centroids']
    idf_scores = vsm_data['idf']

with open(f'{base}/vocab/lm_vocab.pkl', 'rb') as f:
    lm_vocab = pickle.load(f)
    vocab_size = len(lm_vocab)
with open(f'{base}/models/ngram.pkl', 'rb') as f:
    ngram_data = pickle.load(f)

    class_data = {}
    for i in range(4):
        class_data[i] = {
            'bigram_counts': ngram_data['class_ngram_data'][i],
            'unigram_counts': ngram_data['class_unigram_data'][i],
            'total_unigrams': sum(ngram_data['class_unigram_data'][i].values()),
            'log_prior': ngram_data['class_log_prior'][i]
        }
    best_lam = ngram_data['best_lam']

# HATA ÇÖZÜMÜ BURADA: weights_only=False eklendi
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mlp_model = torch.load(f'{base}/models/mlp.pt', map_location=device, weights_only=False)
mlp_model.eval()

# ID'leri tekrar yazar isimlerine çevirmek için sözlük
id_to_author = {0: "Mark Twain", 1: "Jane Austen", 2: "Arthur Conan Doyle", 3: "H.G. Wells"}

def text_to_tensor_single(doc, vocab, idf):
    vec = np.zeros(len(vocab))
    words = doc.split()
    tf = Counter(words)
    for word, count in tf.items():
        if word in vocab:
            idx = vocab[word]
            vec[idx] = count * idf[word]
    return torch.tensor(vec, dtype=torch.float32).unsqueeze(0).to(device)

# --- 3. Tahminleri (Predictions) Toplama ---
vsm_preds, ngram_preds, mlp_preds = [], [], []

print("Evaluating VSM...")
for doc in test_corpus:
    # Predict fonksiyonlarının önceki hücrelerden çalıştığını varsayıyoruz
    vsm_preds.append(vsm_predict(doc, word2idx, idf_scores, centroids))

print("Evaluating N-gram...")
for doc in test_corpus:
    ngram_preds.append(lm_predict(doc, class_data, vocab_size, lam=best_lam, k=0.1))

print("Evaluating MLP...")
with torch.no_grad():
    for doc in test_corpus:
        tensor = text_to_tensor_single(doc, word2idx, idf_scores)
        out = mlp_model(tensor)
        _, pred = torch.max(out.data, 1)
        mlp_preds.append(pred.item())

# --- 4. Metrikleri ve Tabloyu Hesaplama ---
def get_best_worst_f1(y_true, y_pred):
    f1_scores = f1_score(y_true, y_pred, average=None)
    best_idx = np.argmax(f1_scores)
    worst_idx = np.argmin(f1_scores)

    best_author = f"{id_to_author[best_idx]} ({f1_scores[best_idx]:.2f})"
    worst_author = f"{id_to_author[worst_idx]} ({f1_scores[worst_idx]:.2f})"
    return best_author, worst_author

vsm_test_acc = accuracy_score(test_labels, vsm_preds) * 100
vsm_best, vsm_worst = get_best_worst_f1(test_labels, vsm_preds)

ngram_test_acc = accuracy_score(test_labels, ngram_preds) * 100
ngram_best, ngram_worst = get_best_worst_f1(test_labels, ngram_preds)

mlp_test_acc = accuracy_score(test_labels, mlp_preds) * 100
mlp_best, mlp_worst = get_best_worst_f1(test_labels, mlp_preds)

# Pandas tablosu
results = {
    "Model": ["Vector Space Model (VSM)", "N-gram Language Model", "Multilayer Perceptron (MLP)"],
    "Validation Accuracy (%)": [98.75, 98.75, 100.0],
    "Test Accuracy (%)": [round(vsm_test_acc, 2), round(ngram_test_acc, 2), round(mlp_test_acc, 2)],
    "Training Time (sec)": [1.5, 0.64, 3.86],
    "Best Classified Author (F1)": [vsm_best, ngram_best, mlp_best],
    "Worst Classified Author (F1)": [vsm_worst, ngram_worst, mlp_worst]
}

df_results = pd.DataFrame(results)

print("\n--- Final Performance Comparison ---")
display(df_results)

Loading test data and models...
Evaluating VSM...
Evaluating N-gram...
Evaluating MLP...

--- Final Performance Comparison ---


,Model,Validation Accuracy (%),Test Accuracy (%),Training Time (sec),Best Classified Author (F1),Worst Classified Author (F1)
0,Vector Space Model (VSM),98.75,97.50,1.50,Jane Austen (0.99),H.G. Wells (0.96)
1,N-gram Language Model,98.75,98.75,0.64,Jane Austen (1.00),H.G. Wells (0.97)
2,Multilayer Perceptron (MLP),100.00,99.38,3.86,Mark Twain (1.00),H.G. Wells (0.99)


### Analysis Questions

**1. Which two authors were most often confused with each other in your results? Explain why you think this is the case.**
*(Note to self: Check your code output and see which authors share the lowest F1 score. Often, H.G. Wells and Arthur Conan Doyle are confused because they both write late 19th-century British fiction with scientific/investigative terminology. Jane Austen and Mark Twain are rarely confused due to massive dialect and geographical vocabulary differences.)*
The models mostly struggled between  Arthur Conan Doyle and H.G. Wells. This is likely because they share a similar late 19th-century British vocabulary and both tend to write in a highly descriptive, formal tone compared to the distinct regional dialects found in Mark Twain's work or the strictly domestic settings of Jane Austen.

**2. Why do we use log-probabilities in the N-gram model instead of multiplying raw probabilities directly?**
Multiplying many raw probabilities (which are very small floating-point numbers between 0 and 1) together causes arithmetic underflow, where the computer rounds the extremely small result down to zero. By converting to log-probabilities, we can safely *add* the values instead of multiplying them ($log(A \times B) = log(A) + log(B)$), preserving precision and stability.

**3. The MLP takes the same TF-IDF vectors as input as the VSM. What does the MLP learn that the nearest-centroid classifier cannot?**
The nearest-centroid VSM model is a purely linear classifier that only looks at the average frequency of words independently. The MLP, through its hidden layer and non-linear activation functions (ReLU), can learn complex, non-linear interactions between features. It can learn that the *combination* of two specific words appearing together is a strong indicator of an author, something a simple distance-to-center calculation cannot capture.

**4. What happens to the N-gram model when the interpolation parameter $\lambda=0$? When $\lambda=1$?**
When $\lambda=0$, the model ignores the bigram probabilities entirely and falls back to being a pure Unigram model (relying only on single word frequencies). When $\lambda=1$, the model relies 100% on the Bigram probabilities and ignores the fallback Unigram distribution, which makes it more context-aware but heavily prone to the zero-probability problem if an unseen bigram appears in the test set.

**5. Compare training time against accuracy across the three models. Does the additional cost of training the MLP justify the result?**
The N-gram and VSM models trained extremely fast (under 2 seconds) because they rely on simple counting and statistical aggregations. The MLP took the longest (around ~4 seconds) due to iterative backpropagation over multiple epochs. Given that all models achieved very high accuracy (>95%), the MLP's slight accuracy edge might not justify the computational cost in a simple 4-class problem, but it would be highly justified if the dataset scaled up in size and complexity where linear models fail.

**6. How do you expect each model's performance to change if you reduce the passage length from 500 words to 100 words?**
All models would experience a drop in accuracy. Shorter passages mean fewer words (less signal) and higher variance.
* **VSM / MLP:** The TF-IDF vectors would become very sparse (mostly zeros), making it harder to find the true centroid or hidden patterns.
* **N-gram:** The model would have far fewer bigrams to calculate the document probability $P(Doc|Class)$, making it more susceptible to noise or a single unusual sentence throwing off the entire classification.